# Overview
This jupyter notebook encodes a fictional dataset into BIDS format using the "muniverse" framework (found here https://github.com/dfarinagroup/muniverse/ ) 

The experimental setup is intentionally more complex than most regular datasets for illustrative purposes. 
<img src="InputMetadata/experimentalSetup.jpg" alt="Drawing" style="width: 600px;"/> <img src="InputMetadata/schematics.jpg" alt="Drawing" style="width: 400px;"/>

As you can see there are 3 Grids of HDsEMG. There are 2 Grids of HDiEMG (thin filaments). Additionally there are 2 Fine-Wire Electrodes and 2 Needle Electrodes, as well as 2 reference electrodes and 1 ground electrode. Grids have different numbers of rows and columns. 


In [ ]:
import numpy as np
import pandas as pd
import muniverse.utils.bids_routines
import copy

# Metadata 
## Global (dataset wide) metadata
### Subjects data & sidecar
We define the data of our subjects. This will end up in the file "participants.tsv". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/data-summary-files.html#participants-file 

In [ ]:
subjects_data = {
    "participant_id": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06"],
    "age": [42, 43, 44, 45, 46, 47],
    "sex": ["M", "F", "M", "F", "M", "F"],
    "handedness": ["R", "R", "R", "R", "L", "R"],
    "weight": [70, 68, 66, 64, 62, 60],
    "height": [1.7, 1.72, 1.74, 1.76, 1.78, 1.8],
    "group": ["T", "T", "T", "P", "P", "P"],
}

We define the subjects sidecare file. This information will end up in "participants.json". 
We only need to do this for extra columns, since the following defaults are already added by muniverse: "participant_id", "age", "sex", "handedness", "weight", "height". 

In [ ]:
subjects_sidecar = {
    "group": {
        "Description": "Group the subject belongs to.",
        "Levels": {
            "T": "Treatment", 
            "P": "Placebo"
        }
    }
}

### Dataset sidecar 
We define the dataset sidecar. This will end up in "dataset_description.json". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/dataset-description.html#dataset_descriptionjson 

In [ ]:
dataset_sidecar = {
  "Name": "DatasetName",
  "License": "CC BY 4.0",
  "Authors": [
    "alice",
    "bob"
  ], # has to be a list even if it has only one entry 
  "ReferencesAndLinks": [
    "citation of related publication as text",
    "related publication as DOI"
  ], # has to be a list even if it has only one entry 
  "EthicsApprovals": [
    "number of ethics approval."
  ] # has to be a list even if it has only one entry 
}

### Readme File 
We define a readme for the dataset. This becomes "readme.md". More info here: https://bids-specification.readthedocs.io/en/latest/modality-agnostic-files/dataset-description.html#readme 
A readme Template can be found here: https://bids.neuroimaging.io/getting_started/templates/index.html?h=readme#readmemd 


In [ ]:
readme = """
A fictional dataset made up for illustrative purposes. 
The experimental setup is intentionally more complex than most regular datasets for illustrative purposes. 

There are 3 Grids of HDsEMG. There are 2 Grids of HDiEMG (thin filaments). Additionally there are 2 Fine-Wire Electrodes and 2 Needle Electrodes, as well as 2 reference electrodes and 1 ground electrode. Grids have different numbers of rows and columns. 

Other things that should belong in a readme. 
"""

### Tasks
We define the names of our tasks. 

In [ ]:
tasklist = ["taskname1", "taskname2", "taskname3"]

## Local (recording specific) metadata
### EMG sidecar 
We define the emg sidecar. This will end up in files ending with "..._emg.json". There will be one such file per recording. 
In our case they are identical for each recording, except for the *Taskname* and *TaskDescription*. 
More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#sidecar-json-_emgjson 

In [ ]:
emg_sidecar_task1 = {
    "EMGPlacementScheme": "Measured", 
    "EMGPlacementSchemeDescription": "(i) Surface EMG: Four grids were carefully positioned side-to-side with a 4-mm distancebetween the electrodes at the edges of adjacent grids. The 256 electrodes were centered to the muscle belly (right tibialis anterior) and laid  within the muscle perimeter identified through palpation. (ii) Invasive thin film EMG: A guiding filament inserted into a 25-gauge needle was used to place the thin-film structure in the muscle. Afterwards, the needle is withdrawn, leaving only the thin-film array within the muscle. (iii) Invasive Fine Wire EMG: The fine-wires were inserted as pairs using a cannulae to a target depth of 2 cm. After the insertion, the bipolar EMG was visually inspected and if needed the electrode positions were adjusted through light needle movements. Afterwards, the cannula was removed leaving only the fine wires within the muscle. (iv) Invasive Needle EMG: The needle was inserted (perpendicular to the skin) into the proximal part of the ADM targeting an insertion depth of 1 cm. ",
    "EMGReference": "ChannelSpecific",
    "EMGGround": "G1",
    "SamplingFrequency": 2048,
    "PowerLineFrequency": 50,
    "RecordingType": "continuous", 
    "SoftwareFilters": {
        "Anti-aliasing filter": {
            "half-amplitude cutoff (Hz)": 500,
            "Roll-off": "6dB/Octave"
        },
        "some other filter": {
            "filterparameter 1": 500,
            "filterparameter 2": "12dB/Octave",
            "filterparameter 3": 80
        }
    },
    "TaskName": "taskname1",
    "TaskDescription": "Each participant performed two trapezoidal contractions at 30 percent and 50 percent MVC with 120 s of rest in between, consisting of linear ramps up and down performed at 5 percent per second and a plateau maintained for 20 and 15 s at 30 percent and 50 percent MVC, respectively. The order of the contractions was randomized.",
    "Preamplification": 1,
    "Gain": 1,
    "Manufacturer": "some amplifier manufacturer",
    "ManufacturersModelName": "some amplifier model name"
}

# Since almost everything is the same for the other tasks we create a copy and only change TaskName and TaskDescription 
emg_sidecar_task2 = copy.deepcopy(emg_sidecar_task1)
emg_sidecar_task2["TaskName"] = "taskname2"
emg_sidecar_task2["TaskDescription"] = "Participants ate ice cream."

emg_sidecar_task3 = copy.deepcopy(emg_sidecar_task1)
emg_sidecar_task3["TaskName"] = "taskname3"
emg_sidecar_task3["TaskDescription"] = "Participants nodded their head."

# To retrieve these cleanly later from inside a for-loop we pack them into a dict 
emg_sidecar_taskDict = {
    "taskname1": emg_sidecar_task1, 
    "taskname2": emg_sidecar_task2, 
    "taskname3": emg_sidecar_task3, 
}

### Electrode data & sidecar
We load electrode metadata from a file. This becomes "electrodes.tsv". More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#electrodes-description-_electrodestsv 

In [ ]:
el_metadata = pd.read_excel("InputMetadata//electrodes.xlsx")
el_metadata.head()

We define the electrodes sidecar. This becomes "..._electrodes.json" files. Similar to the subjects sidecar we only need to define the non-default columns. 

In [ ]:
electrodes_sidecar = {
    "interelectrodeDistance": "Distance between electrodes. In a grid this means distance between neighboring electrodes. In a fine wire it means distance between the wire tips.", 
    "electrodeSurfaceArea": "Surface area of the electrode", 
    "fineWireDiameter": "Diameter of the fine wire tip", 
    "fineWireRecordingTipLength": "Unisolated length of the fine wire tip", 
    "concentricNeedleDiameter": "Concentric needle diameter", 
    "concentricNeedleLength": "Length of concentric needle", 
    "electrodeManufacturer": "Name of electrode manufacturer", 
    "electrodeManufacturersModelName": "Model name of Electrode", 
    "electrodeType": "Type of electrode", 
}

### Channel data & sidecar
We load channel data from a file. This becomes "..._channels.tsv". More info here: https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electromyography.html#channels-description-_channelstsv 

In [ ]:
ch_metadata = pd.read_excel("InputMetadata//channels.xlsx")
ch_metadata.head()

We define channels sidecar. This becomes "..._channels.json" files. Again we only need to define non-default columns. 

In [ ]:
channels_sidecar = {
    "someAdditionalColumn": "some Description of this column"
}

### Coordinate system sidecar 

forearm
each grid own child system 

In [ ]:
coord_sidecar = {
    "forearmCoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "percent",
        "EMGCoordinateSystemDescription": "x: Radial Styloid Process (RSP) -> Ulnar Styloid Process (USP); y: Right-hand rule (limits: Olecranon Process -> Cubital Fossa); z: midpoint RSP-USP -> Lateral Humerus Epicondyle (LHE)"
    }, 

    "grid1CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": " The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E001",
        "AnchorCoordinates": [30, 50, 80]
    }, 

    "grid2CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E009",
        "AnchorCoordinates": [30, 50, 60]
    }, 

    "grid3CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "E017",
        "AnchorCoordinates": [30, 50, 40]
    }, 

    "intraGrid1CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "iE001",
        "AnchorCoordinates": [30, 50, 70]
    }, 

    "intraGrid2CoordSys" : {
        "EMGCoordinateSystem": "Other",
        "EMGCoordinateUnits": "mm",
        "EMGCoordinateSystemDescription": "The x-axis is left to right, the y-axis is bottom to top. Note: the z-axis is not used.",
        "ParentCoordinateSystem": "forearmCoordSys",
        "AnchorElectrode": "iE009",
        "AnchorCoordinates": [30, 50, 50]
    }
}

In [ ]:
# CoordSystemSidecar = {
#         "grid1": {
#             "EMGCoordinateSystem": "Other", 
#             "EMGCoordinateSystemDescription": "x-axis: proximal -> distal; y-axis: lateral -> medial; all local electrode coordinates (per grid) are mapped on a single global coordinate system.",
#             "EMGCoordinateUnits": "mm",
#             "AnchorElectrode": "E12 - Grid1"
#         },
#         "lowerLeg": {
#             "EMGCoordinateSystem": "Other",
#             "EMGCoordinateSystemDescription": "x-axis: proximal -> distal; y-axis: lateral -> medial; the coordinate system is located at the knee joint", 
#             "EMGCoordinateUnits": "percent"
#         }
#     }

# Building the dataset
Now we create an instance of a muniverse bids_dataset. 
First we set global metadata. 

In [ ]:
FictionalDatasetExample = muniverse.utils.bids_routines.bids_dataset(datasetname="FictionalDatasetExample")
FictionalDatasetExample.set_metadata(field_name='subjects_data', source=subjects_data)
FictionalDatasetExample.set_metadata(field_name='subjects_sidecar', source=subjects_sidecar)
FictionalDatasetExample.set_metadata(field_name='dataset_sidecar', source=dataset_sidecar)
FictionalDatasetExample.readme = readme
FictionalDatasetExample.write()

Then we loop over all participants and tasks, create an instance of bids_emg_recording for each, add all local metadata and the raw data to it. In this case we randomly generate the raw data, but you could easily replace that with a function like *get_raw_data(participant_id, task)* to insert your own data. 

In [ ]:
rng = np.random.default_rng(seed=12345)

# manual override to write only one subject and one task: 
subjects_data["participant_id"] = ["sub-01"]
tasklist = ["taskname1"]

for participant_id in subjects_data["participant_id"]:
    participant_id = participant_id[4:]
    print(f"Bidsifying data of sub-{participant_id}")

    for task in tasklist:
        # create a random array of the correct size to be our raw data. 
        n_channels = len(ch_metadata.loc[:,"name"])
        recordingLength = 1 # seconds 
        samplingFrequency = 2048 # samples per second 
        n_samples = np.ceil(recordingLength * samplingFrequency)
        data = rng.uniform(low=0, high=1, size=(n_channels, n_samples))

        # Make a recording and add data and metadata
        emg_recording = muniverse.utils.bids_routines.bids_emg_recording(
            parent_dataset = FictionalDatasetExample, 
            subject_label=participant_id, 
            task_label=task, 
            datatype='emg',
            inherited_metadata=["coordsystem.json", "electrodes.tsv", "electrodes.json"], 
            inherited_level=["subject", "subject", "dataset"]
        )
        emg_recording.set_metadata(field_name='channels_sidecar', source=channels_sidecar)
        emg_recording.set_metadata(field_name='electrodes_sidecar', source=electrodes_sidecar)
        emg_recording.set_metadata(field_name='emg_sidecar', source=emg_sidecar_taskDict[task])
        emg_recording.set_metadata(field_name='channels', source=ch_metadata)
        emg_recording.set_metadata(field_name='electrodes', source=el_metadata) 

        emg_recording.set_metadata(field_name='coord_sidecar', source=coord_sidecar)

        emg_recording.set_data(field_name='emg_data', mydata=data,fsamp=samplingFrequency)

        # emg_recording.set_metadata(field_name="events_sidecar", source=events_sidecar)
        # emg_recording.set_metadata(field_name="events", source=events)
        emg_recording.write()


# Validation
Finally we use the BIDS-validator to check the integrity of our dataset. 
For more information see here: https://github.com/bids-standard/bids-validator 

In [ ]:
err, warn, _ = FictionalDatasetExample.validate(
    print_errors=True,
    print_warnings=True,
    ignored_codes=[],
    ignored_fields=["HEDVersion"],
    ignored_files=[]
)

print("The BIDS conversion has completed")
print(f"Your BIDS dataset contains {len(err)} errors and {len(warn)} warnings")


In [ ]:

# def get_grid_coordinates(grid_name):
#     """
#     Helper funcion to extract electrode coordinates given a grid type.
    
#     """

#     if grid_name == 'GR04MM1305':
#         x = np.zeros(64)
#         y = np.zeros(64)
#         y[0:12]  = 0
#         x[0:12]  = np.linspace(11*4,0,12)
#         y[12:25] = 4
#         x[12:25] = np.linspace(0,12*4,13)
#         y[25:38] = 8
#         x[25:38] = np.linspace(12*4,0,13)
#         y[38:51] = 12
#         x[38:51] = np.linspace(0,12*4,13)
#         y[51:64] = 16
#         x[51:64] = np.linspace(12*4,0,13)
           
#     else:
#         raise ValueError('The given grid_name has no reference')

#     return(x,y)

# def make_electrode_metadata(
#         ngrids, 
#         gridname='GR04MM1305'
#     ):
#     """
#     Helper function to curate the electrode metadata
    
#     """

#     # Define the columns of the electrode.tsv file
#     columns = ["name", "x", "y", "coordinate_system"]

#     # Init dataframe containing the electrode metadata
#     df = pd.DataFrame(np.nan, index=range(64*ngrids + 2), columns=columns)
#     df = df.astype({
#         "name": "string", 
#         "x": "float", 
#         "y": "float",
#         "coordinate_system": "string", 
#     })


#     # Loop over each electrode (of the four grids) and set metadata
#     elecorode_idx = 0
#     for i in np.arange(ngrids):
#         (xg, yg) = get_grid_coordinates(gridname)
#         for j in np.arange(64):
#             df.loc[elecorode_idx, "name"] = f"E{str(elecorode_idx+1).zfill(3)}" # f'E' + str(elecorode_idx))
#             # Map all electrode coordinates into the grid1 coordinate system
#             df.loc[elecorode_idx, "coordinate_system"] = "grid1"
#             if i==0: # Lateral-Proximal
#                 df.loc[elecorode_idx, "x"] = xg[j]
#                 df.loc[elecorode_idx, "y"] = yg[j]
#             elif i==3: # Medial-Proximal
#                 y_shift = 20 
#                 df.loc[elecorode_idx, "x"] = xg[j]
#                 df.loc[elecorode_idx, "y"] = yg[j] + y_shift
#             elif i==1: # Lateral-Distal
#                 x_shift = 100 
#                 y_shift = 16 
#                 df.loc[elecorode_idx, "x"] = x_shift - xg[j]
#                 df.loc[elecorode_idx, "y"] = y_shift - yg[j]
#             elif i==2: # Medial-Distal
#                 x_shift = 100 
#                 y_shift = 36 
#                 df.loc[elecorode_idx, "x"] = x_shift - xg[j]
#                 df.loc[elecorode_idx, "y"] = y_shift - yg[j]
#             # Take care of the electrode index    
#             elecorode_idx += 1    
   
#    # Add the reference electrodes
#     df.loc[ngrids*64+0, "name"] = "R1"
#     df.loc[ngrids*64+1, "name"] = "R2"

#     df.loc[ngrids*64+0, "coordinate_system"] = "lowerLeg"
#     df.loc[ngrids*64+1, "coordinate_system"] = "lowerLeg"

#     df.loc[ngrids*64+0, "x"] = 90
#     df.loc[ngrids*64+1, "x"] = 95

#     df.loc[ngrids*64+0, "y"] = 0
#     df.loc[ngrids*64+1, "y"] = 0

#     return df

# make_electrode_metadata(ngrids=2)